# 자습 13. 로컬 배포와 운영

> **선택 자습 — 본수업 8시간에 포함하지 않습니다.** 코드 셀은 소형 합성 예제이며 GPU 학습·다운로드·로봇 구동을 자동 실행하지 않습니다. 실제 작업 명령은 설명의 터미널 예시로 제공하며 준비물을 먼저 갖춥니다. TODO는 직접 작성하고 아래 정답·해설로 검증하세요. [전체 자습 경로](selfstudy/README.md)


## 자습 중 막혔을 때 — 오류와 조치

먼저 실패한 셀과 전체 오류를 보존하세요. 아래 문구는 대표 예시이며 버전에 따라 달라질 수 있습니다. TODO의 미완료와 실행 환경의 고장을 구분합니다.

| 증상·오류 | 확인 및 조치 | 재확인 기준 |
|---|---|---|
| `Incomplete local checkpoint` / `Use a new output directory` | 같은 저장점의 전체 checkpoint를 준비하고 기존 결과를 보존해 새 output 이름을 씁니다. | 새 replay.json과 action-chunk.npy 생성 |
| `NPZ keys must exactly match` / dtype 오류 | input_features와 NPZ 키를 대조합니다. 영상 HWC RGB uint8, 상태는 유한한 float32·정확한 차원입니다. | 입력 검사와 저장 영상 확인 통과 |
| `Non-empty saved rename_map` / `Unsupported replay feature` | 기본 도구의 지원 범위 밖입니다. processor를 지우지 말고 명시적인 별도 adapter를 검증합니다. | 변환 전후 의미·규격 일치 |
| gate 거부·오래된 관측·watchdog 정지 | frame ID·시간 기준·단위·한계를 확인합니다. 통과시키려고 검사/정지 기능을 제거하지 않습니다. | 유효 명령 통과, 센서 단절/만료에서 정지 |

**예시에 없는 오류를 확인할 곳:** [VLA 단계별 오류 사전](selfstudy/TROUBLESHOOTING.md)의 13 절과 [준비 안내](selfstudy/README.md). 훈련은 터미널 stdout/stderr와 checkpoint 설정, 재생은 지정 output의 결과 파일을 봅니다.

VLA 훈련·저장 관측 재생에는 noVNC 창이 열리지 않습니다. 본수업 GUI rollout을 다시 실행한 경우에만 FINISHED 후 MuJoCo 창을 닫습니다.

[공통 오류 확인 순서와 문의 양식](../자습_오류해결.md)에서 실행 환경·오류 전체·최소 입력·이미 시도한 조치를 정리하세요. 수정 후에는 실패한 검사부터 다시 통과하고 저장 결과까지 확인합니다.


## 배포는 가중치 파일 복사로 끝나지 않습니다

**완료 결과:** 버전이 고정된 배포 묶음, 저장 입력 재생 결과, 명령 거부 규칙, 재시작·롤백 절차. 소형 실습 90분, 실제 장비 적용은 별도 프로젝트입니다.

![로컬 VLA 배포 구조](../교재그림/vla-selfstudy-deploy.svg)

센서 관측에는 시간과 상태 유효성을 붙입니다. policy worker는 학습 때와 같은 전처리로 행동을 계산하고 후처리합니다. 제어 adapter는 로봇의 단위·좌표계로 변환하고 유효성·수명·범위를 검사합니다. 독립 watchdog이 추론 프로세스 장애에도 장비별 정지 절차를 수행합니다. ROS 토픽은 이 사이의 전송 수단일 뿐 단위·안전·시간 정합성을 자동으로 보장하지 않습니다.

ROS 연결이 필요하면 기존 표준 메시지 또는 버전이 있는 사용자 메시지에 관측 시간, 생성 시간, sequence, frame, action contract version을 담고 소비자가 검사하도록 설계합니다. 본수업 turtlesim의 `/cmd_vel`에 로봇 팔 행동 벡터를 그대로 넣지 않습니다.


## 무엇을 함께 저장하나요?

```text
release-001/
  pretrained_model/    정책 config·safetensors·pre/postprocessor·통계
  cache/               필요한 tokenizer·VLM 설정 등 오프라인 캐시
  contract.json        카메라·state·action의 순서/단위/좌표계/시간 계약
  evaluation.json      조건별 실제 평가와 채택 근거
  environment.txt      코드 commit·패키지·플랫폼·image ID
  SHA256SUMS           묶음 내 파일 해시
```

위 항목은 배포 설계 예시입니다. tokenizer가 별도 캐시에서 로드되는 구현이면 가중치 폴더만 복사해도 인터넷을 요청할 수 있습니다. 인터넷 없는 새 프로세스에서 로딩·입력 재생까지 확인합니다. 학습 재개용 `training_state`는 별도 보관하고 추론 배포물에서 제외할 수 있습니다.

Jetson과 x86의 이미지는 플랫폼별로 만들되 정책·contract·검증 입력의 내용 버전을 맞춥니다. 정책과 processor를 다른 학습 회차에서 섞지 않습니다. 최적화는 기준 PyTorch 결과가 확보된 뒤 수행합니다. ONNX/TensorRT 변환이나 양자화가 이 수업 모델 전체에 자동 지원된다고 가정하지 않으며 변환 후 수치·폐루프 성공·지연을 다시 비교합니다.


## 로봇 없이 저장 입력으로 한 번 추론하기

제공 도구 [selfstudy_replay.py](scripts/selfstudy_replay.py)는 **자신의 로컬 checkpoint와 저장 관측으로 행동 배열만 저장** 합니다. 실제 장비에 명령을 보내지 않습니다. 일반적인 키 이름은 `observation.state`, `observation.images.front`처럼 **학습 dataset과 저장 processor가 기대하는 원래 키** 를 사용합니다. 카메라 배열은 HWC RGB uint8, state는 정규화 전 float32 벡터, task는 별도 문자열입니다. NPZ는 pickle 없이 읽습니다.

```python
# 자신의 수집기에서 준비할 입력 구조 예시. robot/camera 드라이버 구현은 별도입니다.
# np.savez('sample.npz', **{
#     'observation.state': raw_state_float32,
#     'observation.images.front': rgb_hwc_uint8,
# })
```

준비 후 별도 터미널에서 다음처럼 실행합니다. 경로는 실제 개인 자습 루트로 맞추세요. 처음 시험은 GPU를 쓰는 다른 작업이 끝난 뒤 수행합니다.

```bash
bash
export SELFSTUDY_ROOT=/workspace/Data/selfstudy/vla
export HF_HOME="$SELFSTUDY_ROOT/cache/huggingface"
export HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1
python /workspace/10.VLA/scripts/selfstudy_replay.py \
  --checkpoint "$SELFSTUDY_ROOT/train/smoke-001/checkpoints/last/pretrained_model" \
  --observation "$SELFSTUDY_ROOT/replay/sample.npz" \
  --task 'Place the block in the tray.' \
  --output "$SELFSTUDY_ROOT/replay/candidate-001"
```

`action-chunk.npy`와 `replay.json`을 확인합니다. 출력 shape는 해당 checkpoint의 chunk 크기·행동 차원이며 항상 50×7이라고 가정하지 않습니다. 기존 출력은 덮어쓰지 않습니다. 소스의 입력 검사 실패를 차원 자르기나 가짜 카메라로 숨기지 않습니다. 카메라 rename이 있는 checkpoint는 입력 NPZ와 저장 processor를 함께 확인합니다.

이 도구는 `noVNC`를 열지 않습니다. 저장 입력 추론이므로 `FINISHED` 창 닫기도 없습니다. 실제 폐루프와 실물 adapter 검증은 12에서 별도로 수행해야 합니다.


## TODO — 오래되거나 잘못된 행동을 거부하기

아래 함수는 **게이트 로직의 합성 예제** 입니다. 로봇을 정지시키는 코드가 아닙니다. 관측 나이가 0부터 0.2초 이내, frame이 `robot_base`, 값 2개가 모두 유한하고 [-1,1]일 때만 True를 반환하세요. 여기에 쓰는 정규화 범위와 0.2초는 연습 조건이며 자신의 장비에서 측정·정의해야 합니다.


In [ ]:
import math
def accept_action(action, observation_age_s, frame):
    # TODO: 시간·좌표계·길이·유한성·범위 모두 검사
    return None
answer = accept_action([0.1,-0.2], 0.05, 'robot_base')
if answer is None:
    print('TODO 미완료: 행동 승인 조건을 구현하세요.')
else:
    assert answer is True
    assert accept_action([0.1,0.2], 0.3, 'robot_base') is False
    assert accept_action([0.1,0.2], -0.1, 'robot_base') is False
    assert accept_action([float('nan'),0.2], 0.05, 'robot_base') is False
    assert accept_action([2,0.2], 0.05, 'robot_base') is False
    assert accept_action([0.1,0.2], 0.05, 'camera') is False
    assert accept_action([0.1], 0.05, 'robot_base') is False
    print('합성 게이트 검사 통과 — 실제 장비 정지 검증과는 별도')

### 정답과 해설

```python
def accept_action(action, observation_age_s, frame):
    return (
        math.isfinite(observation_age_s) and 0 <= observation_age_s <= 0.2
        and frame == 'robot_base' and len(action) == 2
        and all(math.isfinite(x) and -1 <= x <= 1 for x in action)
    )
```

관측이 오래되면 현재 환경을 설명하지 못할 수 있습니다. 음수 나이는 시계나 timestamp가 잘못됐다는 신호입니다. frame 이름만 검사한다고 좌표 변환이 자동 수행되지는 않으므로 보정과 실제 transform도 별도로 검증합니다. 범위를 넘는 값을 무조건 clip하면 단위가 잘못된 오류를 숨길 수 있어 이 예제는 거부합니다.

실제 구현은 타입·차원, 관절 제한, 충돌·작업 공간, 속도·가속도, sequence 순서, 입력 누락, 통신 장애도 검사해야 합니다. 서로 다른 장비 clock의 timestamp를 그냥 빼지 말고 시간 동기화 또는 같은 clock의 수신 시간을 사용합니다. 오래된 묶음이 늦게 도착해 새 명령을 덮어쓰지 않도록 합니다.

거부 후 행동은 로봇별로 정의합니다. 속도 0, 위치 유지, 제어기 정지 중 무엇이 맞는지 해당 API에서 확인하며 Python 추론 루프가 멈춰도 독립 watchdog이 작동해야 합니다. 실제 작업 공간에서 단계별 시험과 현장 안전 절차를 마련합니다.


## 서비스화와 롤백

<!-- procedure:lesson-49-7099dd:start -->
![배포 묶음을 시험하고 되돌리기](../교재그림/절차/lesson-49.svg)

가중치만 바꾸지 않습니다. 독립 watchdog과 장비별 정지·재개 조건을 함께 확인하세요.
<!-- procedure:lesson-49-7099dd:end -->


센서·정책·제어 루프를 분리하면 오래 걸리는 추론이 센서 수집을 막지 않게 할 수 있습니다. 대신 큐 길이를 제한하고 새 관측·새 명령의 우선순위와 만료 정책을 정해야 합니다. 처음에는 동기식 기준 동작을 확인한 뒤 비동기화하고, 처리량 증가와 제어 안정성을 별도 평가합니다.

새 버전은 **출력만 기록 → 제한된 동작 → 반복 시험 → 채택** 순서로 올립니다. 프로세스 시작 때 모델 hash와 contract version을 로그에 남기고 장치 준비 전에는 actuator를 활성화하지 않습니다. 실패하면 이전 정책·processor·contract 묶음을 함께 복구합니다. 서비스 자동 재시작을 설정해도 로봇 동작까지 자동 재개할지는 별도 조건으로 판단합니다.

**최종 TODO:** 이전 정책이 정상인 장비에 새 정책을 배포했는데 카메라 연결이 끊기면, 어떤 로그가 남고 누가 정지를 수행하며 어느 조건에서 재개할지 작성하세요.

**해설 예시:** 관측 만료·frame 누락 원인과 마지막 정상 sequence를 남기고 독립 제어기 watchdog이 장비별 정지 동작을 수행합니다. 재연결 후 보정·유효 관측·operator 확인을 만족한 뒤 새 episode로 시작합니다. 이전 버전 롤백 시 가중치만 되돌리지 않고 processor와 contract도 함께 바꿉니다.

공식 구현 참고: [LeRobot 0.4.4 policy factory](https://github.com/huggingface/lerobot/blob/v0.4.4/src/lerobot/policies/factory.py), [SmolVLA 구현](https://github.com/huggingface/lerobot/blob/v0.4.4/src/lerobot/policies/smolvla/modeling_smolvla.py). [자습 완료 점검표](selfstudy/README.md)로 돌아가 준비·검증 증거를 정리합니다.
